# 12 — Final ML Model Freeze and Packaging

## Purpose

Freeze and package the **Plan B bridge-type decision-engine models**:

1. `Bauwerksart` classifier
2. `zustandsnote` condition model conditioned on candidate `Bauwerksart`

This notebook creates the reusable model package for later inference.

### Plan B input contract

- latitude
- longitude
- dtv
- bauwerkstoff

### Targets

- classifier → `bauwerksart`
- condition model → `zustandsnote`

### Important model boundary

This notebook packages the **Plan B decision-engine models**. It is distinct from the separate 86-predictor frozen condition model used for the broader bridge-condition/Plan A workflow.

Length and width are therefore not inserted into these frozen model feature lists.

No FEM/InfoCAD or structural design is performed here.


In [1]:
from pathlib import Path
import os
import json
import hashlib
import joblib
import numpy as np
import pandas as pd

from sqlalchemy import create_engine, URL, text
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from getpass import getpass

RANDOM_STATE = 42

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

OUTPUT_DIR = OUTPUT_ROOT / "12_Final_ML_Model_Freeze_and_Packaging"
MODEL_DIR = OUTPUT_DIR / "model_package"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_MANIFEST_PATH = MODEL_DIR / "model_manifest.json"
PACKAGE_INVENTORY_PATH = OUTPUT_DIR / "12_model_package_inventory.csv"
DATA_MANIFEST_PATH = OUTPUT_DIR / "12_data_manifest.txt"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output dir :", OUTPUT_DIR)
print("Model dir  :", MODEL_DIR)


Project root: C:\Datenanalyse\final Project
Dataset root: C:\Datenanalyse\final Project\Dataset_PlanA-B
Output dir : C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging
Model dir  : C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging\model_package


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Item | Source / origin | Transfer method | Role in Notebook 12 | Destination |
|---|---|---|---|---|
| Canonical ML dataset | PostgreSQL `Final_Project` → `final.bridge_ml_dataset_final` | SQL query | Training/package source | In-memory `df` / `work` |
| Coordinates | `geom_x`, `geom_y` in canonical dataset | SQL + EPSG:3857 → EPSG:4326 | latitude/longitude | Model inputs |
| Traffic | `traffic_dtv_mean` | SQL | DTV input | Model inputs |
| Material | `baustoffklasse` | SQL | Material input | Model inputs |
| Bridge type | `bauwerksart_text` | SQL | Classifier target / condition predictor | Model package |
| Condition | `zustandsnote` | SQL | Condition target | Model package |
| Model artifacts | Trained sklearn pipelines | Local file write | Frozen reusable models | `12_Final_ML_Model_Freeze_and_Packaging/model_package` |
| Manifest | Model/package metadata | Generated | Reproducibility and contract | `12_data_manifest.txt`, `model_manifest.json` |
| Package inventory | Actual package files | Generated | Audit | `12_model_package_inventory.csv` |

### Transfer chain

```text
PostgreSQL: final.bridge_ml_dataset_final
                ↓
Notebook 12
   ├── Plan B classifier
   ├── Plan B condition model
   └── reload / deterministic smoke tests
                ↓
12_Final_ML_Model_Freeze_and_Packaging/model_package
```

**No BASt/DWD/Traffic download occurs here.**

**No FEM/InfoCAD calculation or structural design occurs here.**

**The 86-predictor frozen Plan A condition model is a separate model contract and is not replaced by this Plan B package.**


## 01 — Load canonical dataset


In [2]:
## 01 — Load canonical dataset

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(
    url,
    connect_args={
        "connect_timeout": 10,
        "client_encoding": "WIN1252",
    },
)

SOURCE_TABLE = '"final"."bridge_ml_dataset_final"'

with engine.connect() as conn:
    print("PostgreSQL connection: PASS")
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

df = pd.read_sql(
    f"SELECT * FROM {SOURCE_TABLE}",
    engine,
)

print("Canonical dataset:", df.shape)
print("Source table:", SOURCE_TABLE)


PostgreSQL connection: PASS
Database: Final_Project
Canonical dataset: (52214, 97)
Source table: "final"."bridge_ml_dataset_final"


## 02 — Prepare the frozen training contract


In [3]:
required = [
    "bridge_id",
    "geom_x",
    "geom_y",
    "traffic_dtv_mean",
    "baustoffklasse",
    "bauwerksart_text",
    "zustandsnote",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        "Required canonical columns missing: "
        + ", ".join(missing)
    )

# The coordinate columns are already validated in the previous pipeline.
# The final interface uses the geographic coordinates as latitude/longitude.
from pyproj import Transformer

transformer = Transformer.from_crs(
    "EPSG:3857",
    "EPSG:4326",
    always_xy=True,
)

x = pd.to_numeric(df["geom_x"], errors="coerce")
y = pd.to_numeric(df["geom_y"], errors="coerce")

longitude, latitude = transformer.transform(
    x.to_numpy(dtype=float),
    y.to_numpy(dtype=float),
)

work = pd.DataFrame({
    "bridge_id": df["bridge_id"].astype(str),
    "latitude": latitude,
    "longitude": longitude,
    "dtv": pd.to_numeric(
        df["traffic_dtv_mean"],
        errors="coerce",
    ),
    "bauwerkstoff": df["baustoffklasse"].astype("string").str.strip(),
    "bauwerksart": df["bauwerksart_text"].astype("string").str.strip(),
    "zustandsnote": pd.to_numeric(
        df["zustandsnote"],
        errors="coerce",
    ),
}).dropna()

work = work[
    work["zustandsnote"].between(1, 4)
    & work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
].copy()

CLS_FEATURES = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
]

REG_FEATURES = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
    "bauwerksart",
]

print("Usable rows:", f"{len(work):,}")
print("Classifier features:", CLS_FEATURES)
print("Condition-model features:", REG_FEATURES)


Usable rows: 52,214
Classifier features: ['latitude', 'longitude', 'dtv', 'bauwerkstoff']
Condition-model features: ['latitude', 'longitude', 'dtv', 'bauwerkstoff', 'bauwerksart']


## 03 — Freeze candidate bridge-type support

Only bridge types with at least 100 historical observations are included in the final candidate library.


In [4]:
MIN_HISTORICAL_N = 100

candidate_types = (
    work["bauwerksart"]
    .value_counts()
    .loc[lambda s: s >= MIN_HISTORICAL_N]
    .index
    .tolist()
)

if not candidate_types:
    raise RuntimeError("No supported Bauwerksart candidates found.")

print("Candidate bridge types:", len(candidate_types))


Candidate bridge types: 20


## 04 — Train final classifier


In [5]:
cls_pre = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        ["latitude", "longitude", "dtv"],
    ),
    (
        "cat",
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore"),
            ),
        ]),
        ["bauwerkstoff"],
    ),
])

classifier = Pipeline([
    ("preprocess", cls_pre),
    (
        "model",
        ExtraTreesClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
    ),
])

classifier.fit(
    work[CLS_FEATURES],
    work["bauwerksart"],
)

print("Classifier training: PASS")
print("Classifier classes:", len(
    classifier.named_steps["model"].classes_
))


Classifier training: PASS
Classifier classes: 43


## 05 — Train final condition model


In [6]:
reg_pre = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        ["latitude", "longitude", "dtv"],
    ),
    (
        "cat",
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore"),
            ),
        ]),
        ["bauwerkstoff", "bauwerksart"],
    ),
])

condition_model = Pipeline([
    ("preprocess", reg_pre),
    (
        "model",
        ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

condition_model.fit(
    work[REG_FEATURES],
    work["zustandsnote"],
)

print("Condition model training: PASS")


Condition model training: PASS


## 06 — Package models

The preprocessing and estimator are stored together so inference uses exactly the same transformations.


In [7]:
classifier_path = MODEL_DIR / "bridge_type_classifier.joblib"
condition_path = MODEL_DIR / "condition_model.joblib"

joblib.dump(classifier, classifier_path)
joblib.dump(condition_model, condition_path)

print("Saved:", classifier_path.name)
print("Saved:", condition_path.name)


Saved: bridge_type_classifier.joblib
Saved: condition_model.joblib


## 07 — Verify package by reloading


In [8]:
# 07 — Memory-safe package verification

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Recreate model paths
# Reuse the canonical MODEL_DIR created in the project configuration cell.
# Do not recreate a legacy relative path here.
MODEL_DIR = OUTPUT_ROOT / "12_Final_ML_Model_Freeze_and_Packaging" / "model_package"

classifier_path = MODEL_DIR / "bridge_type_classifier.joblib"
condition_path = MODEL_DIR / "condition_model.joblib"

# Check files
assert classifier_path.exists(), f"Missing: {classifier_path}"
assert condition_path.exists(), f"Missing: {condition_path}"

print("Model files found: PASS")


# --------------------------------------------------
# 1. Classifier reload test
# --------------------------------------------------

test_input = pd.DataFrame([{
    "latitude": 50.7374,
    "longitude": 7.0982,
    "dtv": 25000.0,
    "bauwerkstoff": "Stahlbeton",
}])

model_a = joblib.load(
    classifier_path,
    mmap_mode="r"
)

pred_a = model_a.predict_proba(test_input)

classes = model_a.named_steps["model"].classes_

del model_a


model_b = joblib.load(
    classifier_path,
    mmap_mode="r"
)

pred_b = model_b.predict_proba(test_input)

assert np.allclose(
    pred_a,
    pred_b,
    rtol=0,
    atol=1e-12
)

top_type = classes[np.argmax(pred_a[0])]

del model_b

print("Classifier reload verification: PASS")
print("Top predicted bridge type:", top_type)


# --------------------------------------------------
# 2. Condition-model reload test
# --------------------------------------------------

condition_input = test_input.copy()
condition_input["bauwerksart"] = top_type


condition_a = joblib.load(
    condition_path,
    mmap_mode="r"
)

reg_a = condition_a.predict(
    condition_input[
        [
            "latitude",
            "longitude",
            "dtv",
            "bauwerkstoff",
            "bauwerksart",
        ]
    ]
)

del condition_a


condition_b = joblib.load(
    condition_path,
    mmap_mode="r"
)

reg_b = condition_b.predict(
    condition_input[
        [
            "latitude",
            "longitude",
            "dtv",
            "bauwerkstoff",
            "bauwerksart",
        ]
    ]
)

assert np.allclose(
    reg_a,
    reg_b,
    rtol=0,
    atol=1e-12
)

print("Condition-model reload verification: PASS")
print("Predicted zustandsnote:", float(reg_b[0]))

Model files found: PASS
Classifier reload verification: PASS
Top predicted bridge type: Plattenbalkenbrücke, Trägerrostbrücke
Condition-model reload verification: PASS
Predicted zustandsnote: 2.388649591391813


## 08 — Smoke test with the agreed four-input interface


In [9]:
# 08 — Smoke test of the final 4-input interface

import numpy as np
import pandas as pd
import joblib

from pathlib import Path

# Reuse the canonical MODEL_DIR created in the project configuration cell.
MODEL_DIR = OUTPUT_ROOT / "12_Final_ML_Model_Freeze_and_Packaging" / "model_package"

classifier_path = MODEL_DIR / "bridge_type_classifier.joblib"
condition_path = MODEL_DIR / "condition_model.joblib"

assert classifier_path.exists()
assert condition_path.exists()

# Final user inputs
SMOKE_TEST = pd.DataFrame([{
    "latitude": 50.7374,
    "longitude": 7.0982,
    "dtv": 25000.0,
    "bauwerkstoff": "Stahlbeton",
}])

# Load classifier
classifier = joblib.load(
    classifier_path,
    mmap_mode="r"
)

proba = classifier.predict_proba(SMOKE_TEST)[0]
classes = classifier.named_steps["model"].classes_

smoke_classification = pd.DataFrame({
    "bauwerksart": classes,
    "classification_probability": proba,
})

smoke_classification = (
    smoke_classification
    .sort_values(
        "classification_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

display(smoke_classification.head(10))

# Top candidate types
top_types = smoke_classification.head(10)["bauwerksart"].tolist()

del classifier


# Condition model
condition_model = joblib.load(
    condition_path,
    mmap_mode="r"
)

smoke_condition_input = pd.DataFrame({
    "latitude": [SMOKE_TEST.loc[0, "latitude"]] * len(top_types),
    "longitude": [SMOKE_TEST.loc[0, "longitude"]] * len(top_types),
    "dtv": [SMOKE_TEST.loc[0, "dtv"]] * len(top_types),
    "bauwerkstoff": [SMOKE_TEST.loc[0, "bauwerkstoff"]] * len(top_types),
    "bauwerksart": top_types,
})

REG_FEATURES = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
    "bauwerksart",
]

smoke_condition_input["predicted_zustandsnote"] = np.clip(
    condition_model.predict(
        smoke_condition_input[REG_FEATURES]
    ),
    1,
    4,
)

display(smoke_condition_input)

del condition_model

print("12 STATUS: COMPLETE")

,bauwerksart,classification_probability
0,"Plattenbalkenbrücke, Trägerrostbrücke",0.236000
1,Gewölbe- bzw. Bogenbrücke,0.110000
2,Plattenbrücke,0.100000
3,Balkenbrücke / Mittelträger / Trapezplatte,0.096000
4,Brücke mit Balken- / Plattenmischsystem,0.086000
5,Bogenbrücke mit Bogenscheiben,0.066000
6,Hohlkastenbrücke,0.060000
7,"Rohr als Brücke, ohne Ummantelung",0.058000
8,Gewölbe-/Bogenbrücke ohne Aufbeton,0.047672
9,Brücke als geschlossener Rahmen,0.030328


,latitude,longitude,dtv,bauwerkstoff,bauwerksart,predicted_zustandsnote
0,50.7374,7.0982,25000.0,Stahlbeton,"Plattenbalkenbrücke, Trägerrostbrücke",2.388650
1,50.7374,7.0982,25000.0,Stahlbeton,Gewölbe- bzw. Bogenbrücke,2.152906
2,50.7374,7.0982,25000.0,Stahlbeton,Plattenbrücke,2.109521
3,50.7374,7.0982,25000.0,Stahlbeton,Balkenbrücke / Mittelträger / Trapezplatte,2.072671
4,50.7374,7.0982,25000.0,Stahlbeton,Brücke mit Balken- / Plattenmischsystem,2.266340
5,50.7374,7.0982,25000.0,Stahlbeton,Bogenbrücke mit Bogenscheiben,2.309876
6,50.7374,7.0982,25000.0,Stahlbeton,Hohlkastenbrücke,2.422264
7,50.7374,7.0982,25000.0,Stahlbeton,"Rohr als Brücke, ohne Ummantelung",1.943968
8,50.7374,7.0982,25000.0,Stahlbeton,Gewölbe-/Bogenbrücke ohne Aufbeton,2.173010
9,50.7374,7.0982,25000.0,Stahlbeton,Brücke als geschlossener Rahmen,2.045314


12 STATUS: COMPLETE


## 09 — Create model manifest


In [10]:
## 09 — Final Model Manifest

CLS_FEATURES = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
]

REG_FEATURES = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
    "bauwerksart",
]

manifest = {
    "stage": 12,
    "status": "FROZEN",
    "package_scope": "Plan B bridge-type decision engine",
    "random_state": RANDOM_STATE,
    "source_table": SOURCE_TABLE,

    "classifier": {
        "file": classifier_path.name,
        "algorithm": "ExtraTreesClassifier",
        "target": "bauwerksart",
        "features": CLS_FEATURES,
    },

    "condition_model": {
        "file": condition_path.name,
        "algorithm": "ExtraTreesRegressor",
        "target": "zustandsnote",
        "features": REG_FEATURES,
    },

    "candidate_support_rule": {
        "minimum_historical_n": MIN_HISTORICAL_N,
        "candidate_count": len(candidate_types),
    },

    "model_contract": {
        "length_used_by_ml": False,
        "width_used_by_ml": False,
        "fem_input_to_ml": False,
        "structural_design": False,
        "infoCAD": False,
    },

    "separate_plan_a_model": {
        "exists": True,
        "predictor_count": 86,
        "role": "broader bridge-condition workflow",
        "not_replaced_by_this_package": True,
    },
}

MODEL_MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Manifest saved:", MODEL_MANIFEST_PATH)


Manifest saved: C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging\model_package\model_manifest.json


## 10 — Final freeze status

After this notebook, the model package is treated as frozen.

Future inference must load these files rather than silently changing the model.


In [11]:
print("12 STATUS: COMPLETE")
print("Classifier: FROZEN")
print("Condition model: FROZEN")
print("Input contract: latitude + longitude + dtv + bauwerkstoff")
print("Bridge type target: bauwerksart")
print("Condition evidence: zustandsnote")
print("FEM / InfoCAD: EXCLUDED")
print("Length / width: EXCLUDED")


12 STATUS: COMPLETE
Classifier: FROZEN
Condition model: FROZEN
Input contract: latitude + longitude + dtv + bauwerkstoff
Bridge type target: bauwerksart
Condition evidence: zustandsnote
FEM / InfoCAD: EXCLUDED
Length / width: EXCLUDED


## 10 — Final freeze and package audit


In [12]:
classifier_hash = hashlib.sha256(classifier_path.read_bytes()).hexdigest()
condition_hash = hashlib.sha256(condition_path.read_bytes()).hexdigest()
manifest_hash = hashlib.sha256(MODEL_MANIFEST_PATH.read_bytes()).hexdigest()

package_inventory = pd.DataFrame([
    {
        "artifact": classifier_path.name,
        "path": str(classifier_path),
        "exists": classifier_path.exists(),
        "size_bytes": classifier_path.stat().st_size,
        "sha256": classifier_hash,
        "role": "Plan B Bauwerksart classifier",
    },
    {
        "artifact": condition_path.name,
        "path": str(condition_path),
        "exists": condition_path.exists(),
        "size_bytes": condition_path.stat().st_size,
        "sha256": condition_hash,
        "role": "Plan B zustandsnote condition model",
    },
    {
        "artifact": MODEL_MANIFEST_PATH.name,
        "path": str(MODEL_MANIFEST_PATH),
        "exists": MODEL_MANIFEST_PATH.exists(),
        "size_bytes": MODEL_MANIFEST_PATH.stat().st_size,
        "sha256": manifest_hash,
        "role": "Frozen model contract",
    },
])

display(package_inventory)

package_inventory.to_csv(
    PACKAGE_INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)

DATA_MANIFEST_PATH.write_text(
    "Notebook 12 — Final ML Model Freeze and Packaging\n"
    "================================================\n\n"
    f"PROJECT_ROOT: {PROJECT_ROOT}\n"
    f"SOURCE: PostgreSQL {SOURCE_TABLE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n"
    f"MODEL_DIR: {MODEL_DIR}\n\n"
    "PACKAGE SCOPE: Plan B bridge-type decision engine\n"
    "CLASSIFIER FEATURES: latitude, longitude, dtv, bauwerkstoff\n"
    "CONDITION FEATURES: latitude, longitude, dtv, bauwerkstoff, bauwerksart\n"
    "TARGETS: bauwerksart, zustandsnote\n"
    "LENGTH/WIDTH IN THESE MODELS: NO\n"
    "FEM/InfoCAD: NO\n"
    "STRUCTURAL DESIGN: NO\n\n"
    "SEPARATE PLAN A CONDITION MODEL: 86 predictors; not replaced by this package.\n\n"
    f"CLASSIFIER SHA256: {classifier_hash}\n"
    f"CONDITION MODEL SHA256: {condition_hash}\n"
    f"MODEL MANIFEST SHA256: {manifest_hash}\n\n"
    f"PACKAGE INVENTORY: {PACKAGE_INVENTORY_PATH}\n",
    encoding="utf-8",
)

assert classifier_path.exists()
assert condition_path.exists()
assert MODEL_MANIFEST_PATH.exists()

print("12 STATUS: COMPLETE")
print("Classifier: FROZEN")
print("Condition model: FROZEN")
print("Package scope: Plan B decision engine")
print("Output directory:", OUTPUT_DIR)
print("Data manifest:", DATA_MANIFEST_PATH)


,artifact,path,exists,size_bytes,sha256,role
0,bridge_type_classifier.joblib,C:\Datenanalyse\final Project\Output_PlanA-B\1...,True,13430703187,28fff172bdbbab2de96b5273f3c02d62d09f0ac5e790f0...,Plan B Bauwerksart classifier
1,condition_model.joblib,C:\Datenanalyse\final Project\Output_PlanA-B\1...,True,350505539,e47637fbe76018ec76fc2852fedb583439e56077f72fc9...,Plan B zustandsnote condition model
2,model_manifest.json,C:\Datenanalyse\final Project\Output_PlanA-B\1...,True,1146,ff6a18736fc412023fe517dc8e9fd31a1154de6833ac29...,Frozen model contract


12 STATUS: COMPLETE
Classifier: FROZEN
Condition model: FROZEN
Package scope: Plan B decision engine
Output directory: C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging\12_data_manifest.txt
